In [0]:
from pyspark.sql.functions import explode, col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Paths to Delta tables
users_path = "/Volumes/project_2/datalake/landing_zone/users/"
transactions_path = "/Volumes/project_2/datalake/landing_zone/transactions/"
customers_path = "/Volumes/project_2/datalake/landing_zone/customers/"
products_path = "/Volumes/project_2/datalake/landing_zone/products/"
silver_dir = "/Volumes/project_2/datalake/silver/"

# Load Delta tables directly from directories
users_df = spark.read.format("delta").load(users_path)
transactions_df = spark.read.format("delta").load(transactions_path)

# Clean and flatten users
stg_user_events = users_df.select(
    "event_id",
    "user_id",
    "session_id",
    "event_type",
    "timestamp",
    "page",
    "device",
    "browser",
    "country",
    "city",
    "search_query",
    "element_id",
    "product_id",
    "quantity"
)

# Define schema for line item JSON
line_item_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True)
])

# Clean and flatten transactions
stg_transactions = transactions_df.select(
    "transaction_id",
    "user_id",
    "transaction_type",
    "timestamp",
    "status",
    "currency",
    "payment_method",
    "line_items",
    "total",
    "billing_address",
    "shipping_address"
).withColumn("line_item", explode("line_items")).drop("line_items") \
 .withColumn("line_item_parsed", from_json(col("line_item"), line_item_schema)) \
 .select(
    "transaction_id",
    "user_id",
    "transaction_type",
    "timestamp",
    "status",
    "currency",
    "payment_method",
    col("line_item_parsed.product_id").alias("product_id"),
    col("line_item_parsed.product_name").alias("product_name"),
    col("line_item_parsed.category").alias("category"),
    col("line_item_parsed.brand").alias("brand"),
    col("line_item_parsed.quantity").alias("quantity"),
    col("line_item_parsed.unit_price").alias("unit_price"),
    col("billing_address.street").alias("billing_street"),
    col("billing_address.city").alias("billing_city"),
    col("billing_address.state").alias("billing_state"),
    col("billing_address.zip_code").alias("billing_zip_code"),
    col("billing_address.country").alias("billing_country"),
    col("shipping_address.street").alias("shipping_street"),
    col("shipping_address.city").alias("shipping_city"),
    col("shipping_address.state").alias("shipping_state"),
    col("shipping_address.zip_code").alias("shipping_zip_code"),
    col("shipping_address.country").alias("shipping_country"),
    "total"
)

# Save user events and transactions to silver_dir as Delta tables
stg_user_events.write.format("delta").mode("overwrite").save(f"{silver_dir}user_events")
stg_transactions.write.format("delta").mode("overwrite").save(f"{silver_dir}transactions")

# Output user events and transactions
display(stg_user_events)
display(stg_transactions)